## Keycloak: 모든 유저의 sn / givenName 조회

이 노트북은 Keycloak Admin REST API를 이용해 **모든 유저의 `sn`, `givenName`** 를 수집하고,
`sn`에 공백이 포함되는 등 **풀네임이 들어갔을 가능성이 있는 케이스**를 따로 추려 CSV로 저장합니다.

- **변경(수정/업데이트)은 하지 않습니다.** (조회 + 리포트만)
- 인증정보는 **환경변수 또는 실행 시 입력(getpass)** 으로 받습니다.

### 필요한 정보
- `KEYCLOAK_BASE_URL` 예: `http://192.168.2.59:8080`
- `KEYCLOAK_REALM` 예: `sso`
- `KEYCLOAK_CLIENT_ID` 예: `admin-cli`
- `KEYCLOAK_USERNAME`
- `KEYCLOAK_PASSWORD`

> 토큰: `BASE_URL/realms/{REALM}/protocol/openid-connect/token`
> 유저목록: `BASE_URL/admin/realms/{REALM}/users`


In [ ]:
# 필요하면 설치 (이미 설치돼있으면 스킵)
%pip -q install requests pandas


In [ ]:
import os
from getpass import getpass
from typing import Any, Dict, List, Optional, Tuple

import requests
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)


In [ ]:
def _env_or_prompt(name: str, *, secret: bool = False, default: Optional[str] = None) -> str:
    v = os.getenv(name)
    if v:
        return v
    if default is not None and not secret:
        prompt = f"{name} [{default}]: "
        inp = input(prompt).strip()
        return inp or default
    if secret:
        return getpass(f"{name}: ")
    return input(f"{name}: ").strip()


def get_admin_token(
    base_url: str,
    realm: str,
    client_id: str,
    username: str,
    password: str,
    *,
    verify_tls: bool = True,
    timeout: int = 30,
) -> str:
    token_url = f"{base_url.rstrip('/')}/realms/{realm}/protocol/openid-connect/token"
    data = {
        "grant_type": "password",
        "client_id": client_id,
        "username": username,
        "password": password,
    }
    r = requests.post(token_url, data=data, timeout=timeout, verify=verify_tls)
    r.raise_for_status()
    j = r.json()
    if "access_token" not in j:
        raise RuntimeError(f"No access_token in response: {j}")
    return j["access_token"]


def fetch_all_users(
    base_url: str,
    realm: str,
    token: str,
    *,
    page_size: int = 100,
    verify_tls: bool = True,
    timeout: int = 30,
) -> List[Dict[str, Any]]:
    # Keycloak은 /users?first=0&max=100 같은 방식으로 페이징
    users_url = f"{base_url.rstrip('/')}/admin/realms/{realm}/users"
    headers = {"Authorization": f"Bearer {token}"}

    all_users: List[Dict[str, Any]] = []
    first = 0
    while True:
        params = {"first": first, "max": page_size}
        r = requests.get(users_url, headers=headers, params=params, timeout=timeout, verify=verify_tls)
        r.raise_for_status()
        batch = r.json()
        if not isinstance(batch, list):
            raise RuntimeError(f"Unexpected users response: {batch}")
        if not batch:
            break
        all_users.extend(batch)
        first += len(batch)
    return all_users
